# QC-Py-31 - Transformer Encoder Multi-Asset (GPU)

[< Retour au sommaire](../README.md) | [Suivant : DQN Training >](QC-Py-32-DQN-Training.ipynb)

**Module** : Machine Learning pour le Trading - Modeles Transformer  
**Pre-requis** : QC-Py-30 LSTM Training  
**Duree estimee** : 45 minutes  

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :

1. Implementer un **encodeur Transformer natif (PyTorch)** pour les series temporelles financieres
2. Integrer des **features macro** (VIX, courbe des taux) dans le modele
3. Comprendre le **mecanisme d'auto-attention multi-tete** applique aux marches
4. **Comparer** les performances Transformer vs LSTM (QC-Py-30)
5. **Deployer** le modele dans une strategie QuantConnect

### Prerequis

- Notebook QC-Py-30 LSTM Training complete
- Comprehension de PyTorch (tenseurs, modules, entrainement)
- GPU recommande (CUDA) mais CPU possible

### Structure du notebook

| Section | Sujet | Duree |
|---------|-------|-------|
| 1 | Configuration PyTorch et GPU | 3 min |
| 2 | Chargement des donnees (30 stocks + macro) | 5 min |
| 3 | Feature engineering avec features macro | 5 min |
| 4 | Split temporel et normalisation | 3 min |
| 5 | Modele Transformer Encoder | 5 min |
| 6 | Entrainement GPU | 8 min |
| 7 | Courbes d'apprentissage | 2 min |
| 8 | Evaluation sur le set de test | 2 min |
| 9 | Visualisation des predictions | 2 min |
| 10 | Visualisation de l'attention | 3 min |
| 11 | Backtest simplifie | 3 min |
| 12 | Sauvegarde du modele | 2 min |
| 13 | Integration QuantConnect Cloud | 3 min |
| 14 | Resume et conclusions | 2 min |

> **[REFERENCE QC Cloud]**
> Ce notebook illustre du code QuantConnect a executer dans l'IDE Cloud (https://www.quantconnect.com/research).
> L'environnement local ne dispose pas de QuantBook ni de l'historical data feed.
> Pour executer : cloner le projet QC associe, ouvrir `research.ipynb`, executer cellule par cellule.

---

> **Mode d'emploi** : Ce notebook a **deux parties** :
> 1. **Sections analyse/ML** (pandas, sklearn, matplotlib) : **executables en Jupyter local**
> 2. **Sections integration QC** (classes `QCAlgorithm`) : **code de reference a copier dans `main.py`** de votre projet QC Lab
>
> Les cellules QCAlgorithm sont marquees `# [REFERENCE QC]` et ne sont pas executables localement.
> Voir le guide : [ECE-QC-QUICKSTART.md](../ECE-QC-QUICKSTART.md)

---

---

## Section 1 : Configuration PyTorch et GPU

### Pourquoi le Transformer pour la finance ?

Le mecanisme d'auto-attention du Transformer presente des avantages specifiques pour les series temporelles financieres :

| Avantage | Description |
|----------|-------------|
| **Attention globale** | Chaque timestep peut directement se referer a n'importe quel autre timestep |
| **Parallelisme** | Contrairement aux LSTM, tous les timesteps sont traites en parallele |
| **Interpretabilite** | Les poids d'attention revelent quelles periodes historiques influencent la prediction |
| **Multi-tete** | Chaque tete peut capturer un type de dependance different (tendance, volatilite, correlation) |

### Architecture cible

```
Input [batch, seq_len, features]
         |
    [Input Projection] -> d_model=256
         |
    [Positional Encoding] -> sinusoidal
         |
    [TransformerEncoder x 4 layers]
    - MultiHead Attention (nhead=8)
    - FeedForward (dim=1024)
    - LayerNorm + Residual
         |
    [Global Average Pooling]
         |
    +------->[Regression Head] -> forward return
    |
    +------->[Classification Head] -> direction (up/down)
```

Le modele utilise `nn.TransformerEncoder` et `nn.TransformerEncoderLayer` de PyTorch natif, avec un encodage positionnel sinusooidal classique.

In [1]:
# Imports standards
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
import time
import math
import os
warnings.filterwarnings('ignore')

# Configuration matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Seed pour reproductibilite
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# --- Hyperparametres ---
SEQ_LEN = 60          # Fenetre d'entree (3 mois de trading)
PRED_LEN = 5           # Horizon de prediction (1 semaine)
D_MODEL = 256          # Dimension du Transformer
NHEAD = 8              # Nombre de tetes d'attention
NUM_LAYERS = 4         # Couches de l'encodeur
DIM_FEEDFORWARD = 1024 # Dimension FFN
BATCH_SIZE = 32        # Taille de batch (thermal-safe pour GPU laptop)
EPOCHS = 25            # Nombre d'epochs (reduit pour securite thermique)
PATIENCE = 7           # Patience pour early stopping
LR = 0.0005            # Learning rate
DROPOUT = 0.1          # Dropout
TRAIN_RATIO = 0.70     # Split train
VAL_RATIO = 0.15       # Split validation
N_STOCKS = 30          # Nombre d'actions (spec C2.2)

print(f"\nHyperparametres:")
print(f"  Sequence: {SEQ_LEN} jours -> Prediction: {PRED_LEN} jours")
print(f"  Transformer: d_model={D_MODEL}, nhead={NHEAD}, layers={NUM_LAYERS}")
print(f"  FFN dim: {DIM_FEEDFORWARD}, Dropout: {DROPOUT}")
print(f"  Batch size: {BATCH_SIZE}, Epochs: {EPOCHS}, LR: {LR}")

PyTorch version: 2.11.0+cu128
Device: cuda
GPU: NVIDIA GeForce RTX 3080 Ti Laptop GPU
VRAM: 17.2 GB

Hyperparametres:
  Sequence: 60 jours -> Prediction: 5 jours
  Transformer: d_model=256, nhead=8, layers=4
  FFN dim: 1024, Dropout: 0.1
  Batch size: 32, Epochs: 25, LR: 0.0005


### Interpretation : Configuration

La cellule configure l'environnement d'entrainement avec les parametres cles :

| Parametre | Valeur | Justification |
|-----------|--------|---------------|
| `SEQ_LEN` | 60 | Environ 3 mois de trading, capture les tendances a moyen terme |
| `PRED_LEN` | 5 | 1 semaine de prediction, equilibre precision et horizon |
| `D_MODEL` | 256 | Dimension suffisante pour 30+ features sans bottleneck |
| `NHEAD` | 8 | 256/8 = 32 par tete, permet de capturer differentes correlations |
| `NUM_LAYERS` | 4 | Profondeur suffisante sans surparametrisation |
| `BATCH_SIZE` | 32 | Thermal-safe pour GPU laptop (evite le throttling) |
| `EPOCHS` | 25 | Reduit par rapport au LSTM pour limiter le temps GPU |

**Points cles** :
1. Le GPU accelere considerablement l'entrainement du Transformer (operations matricielles parallles)
2. Le batch size de 32 est un compromis entre stabilite du gradient et securite thermique
3. Le learning rate 0.0005 est plus faible que le LSTM car le Transformer est plus sensible au LR

---

## Section 2 : Chargement des donnees (30 stocks + macro)

### Approche multi-actifs

Nous telechargeons les donnees de 30 actions SP500 (top par capitalisation boursiere) ainsi que des indicateurs macroeconomiques :

- **VIX** (^VIX) : Index de volatilite implicite, indicateur de peur du marche
- **Taux 10 ans** (^TNX) : Rendement des Treasury 10 ans
- **Taux 30 ans** (^TYX) : Rendement des Treasury 30 ans

Le spread 10Y-2Y sera calcule comme proxy de la courbe des taux (un spread negatif = inversion de courbe, signal de recession).

In [2]:
# Telechargement des donnees via yfinance
import yfinance as yf

# 30 top SP500 par capitalisation boursiere
TICKERS_30 = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'BRK-B',
    'JPM', 'V', 'UNH', 'JNJ', 'WMT', 'XOM', 'MA', 'PG', 'HD', 'CVX',
    'MRK', 'ABBV', 'AVGO', 'KO', 'PEP', 'COST', 'ADBE', 'CRM', 'AMD',
    'NFLX', 'TMO', 'CSCO'
]

# Indices macro
MACRO_TICKERS = ['^VIX', '^TNX', '^TYX']

# Periode d'etude
START = '2014-01-01'
END = '2025-01-01'

print(f"Telechargement de {len(TICKERS_30)} actions + {len(MACRO_TICKERS)} indicateurs macro...")
print(f"Periode: {START} a {END}")

# Telecharger toutes les donnees en une seule requete
raw = yf.download(
    TICKERS_30 + MACRO_TICKERS,
    start=START, end=END,
    auto_adjust=True
)

print(f"\nDonnees telechargees: {raw.shape}")
print(f"Colonnes (niveaux): {[str(c) for c in raw.columns[:5]]}...")

# Extraire les prix de cloture (gestion MultiIndex yfinance >=0.2.x)
if isinstance(raw.columns, pd.MultiIndex):
    close_all = raw['Close']
else:
    close_all = raw[['Close']]

# Verifier les donnees manquantes
print(f"\nForme du DataFrame Close: {close_all.shape}")
print(f"Colonnes disponibles: {len(close_all.columns)}")

# Compter les NaN par colonne
nan_counts = close_all.isna().sum()
if nan_counts.sum() > 0:
    print(f"\nColonnes avec NaN:")
    for col, count in nan_counts[nan_counts > 0].items():
        print(f"  {col}: {count} NaN ({count/len(close_all)*100:.1f}%)")

# Remplir les NaN par forward fill puis backward fill
close_all = close_all.ffill().bfill()

print(f"\nDonnees nettoyees: {close_all.shape}")
print(f"NaN restants: {close_all.isna().sum().sum()}")
print(f"\nApercu des 5 premieres lignes:")
print(close_all[TICKERS_30[:5]].head())

Telechargement de 30 actions + 3 indicateurs macro...
Periode: 2014-01-01 a 2025-01-01


[                       0%                       ]

[***                    6%                       ]  2 of 33 completed

[***                    6%                       ]  2 of 33 completed

[*******               15%                       ]  5 of 33 completed

...

[**********************88%*****************      ]  29 of 33 completed

[**********************94%********************   ]  31 of 33 completed

[**********************97%********************** ]  32 of 33 completed

[*********************100%***********************]  33 of 33 completed


Donnees telechargees: (2768, 165)
Colonnes (niveaux): ["('Close', 'AAPL')", "('Close', 'ABBV')", "('Close', 'ADBE')", "('Close', 'AMD')", "('Close', 'AMZN')"]...

Forme du DataFrame Close: (2768, 33)
Colonnes disponibles: 33

Colonnes avec NaN:
  ^TNX: 1 NaN (0.0%)
  ^TYX: 1 NaN (0.0%)

Donnees nettoyees: (2768, 33)
NaN restants: 0

Apercu des 5 premieres lignes:
Ticker           AAPL       MSFT      GOOGL       AMZN      NVDA
Date                                                            
2014-01-02  17.140665  30.760920  27.627516  19.898500  0.373844
2014-01-03  16.764149  30.553963  27.425978  19.822001  0.369365
2014-01-06  16.855572  29.908289  27.731758  19.681499  0.374315
2014-01-07  16.735025  30.140072  28.266380  19.901501  0.380444
2014-01-08  16.841002  29.602007  28.325201  20.096001  0.385630


### Interpretation : Chargement des donnees

Le telechargement recupere plus de 10 ans de donnees pour 30 actions et 3 indicateurs macro.

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| Actions | 30 tickers SP500 | Couverture large du marche US |
| Macro | VIX, TNX, TYX | Sentiment et taux directeurs |
| Periode | 2014-2025 | Plus de 10 ans, incluant crises majeures |
| Frequence | Quotidienne | Adapted au style swing/position trading |

**Points cles** :
1. Les donnees sont nettoyees via forward-fill puis backward-fill pour gerer les jours de non-trading
2. Le VIX est un indicateur avance de volatilite -- les pics de VIX precedent souvent les baisses de marche
3. Le spread de taux (10Y-30Y) contient de l'information sur les anticipations de croissance
4. La gestion du MultiIndex yfinance (version >= 0.2.x) necessite un acces specifique au niveau 'Close'

---

## Section 3 : Feature engineering avec features macro

### Features par action vs features macro

Nous distinguons deux types de features :

| Type | Features | Partage |
|------|----------|--------|
| **Par action** | ret_1d, ret_5d, ret_20d, vol_10d, vol_20d, momentum, rsi | Specifique a chaque action |
| **Macro** | vix_level, vix_change, yield_spread, yield_curve_slope | Partage entre toutes les actions |

Les features macro sont concatenees aux features de chaque action a chaque timestep. Cela permet au Transformer d'apprendre comment le contexte macro influence chaque action individuellement.

In [3]:
def compute_features(close_all, tickers, macro_tickers):
    """
    Calcule les features pour chaque action et les features macro.
    
    Parameters
    ----------
    close_all : pd.DataFrame
        Prix de cloture de tous les tickers
    tickers : list
        Liste des tickers d'actions
    macro_tickers : list
        Liste des tickers macro (^VIX, ^TNX, ^TYX)
    
    Returns
    -------
    dict : {ticker: pd.DataFrame avec features}
    list : noms des colonnes de features
    """
    all_features = {}
    
    # --- Features macro (partagees entre toutes les actions) ---
    vix = close_all['^VIX'] if '^VIX' in close_all.columns else pd.Series(20, index=close_all.index)
    tnx = close_all['^TNX'] if '^TNX' in close_all.columns else pd.Series(3, index=close_all.index)
    tyx = close_all['^TYX'] if '^TYX' in close_all.columns else pd.Series(4, index=close_all.index)
    
    macro_df = pd.DataFrame(index=close_all.index)
    macro_df['vix_level'] = vix
    macro_df['vix_change'] = vix.pct_change()
    macro_df['yield_spread'] = tnx - tyx  # 10Y - 30Y spread
    macro_df['yield_curve_slope'] = tnx.diff(20)  # Variation 20 jours du 10Y
    
    # Remplir les NaN macro
    macro_df = macro_df.ffill().bfill().fillna(0)
    
    # --- Features par action ---
    valid_tickers = [t for t in tickers if t in close_all.columns]
    
    for ticker in valid_tickers:
        prices = close_all[ticker].copy()
        
        df = pd.DataFrame(index=prices.index)
        df['close'] = prices
        
        # Rendements a differentes echelles
        df['ret_1d'] = prices.pct_change(1)
        df['ret_5d'] = prices.pct_change(5)
        df['ret_20d'] = prices.pct_change(20)
        
        # Volatilite realisee
        df['vol_10d'] = df['ret_1d'].rolling(10).std()
        df['vol_20d'] = df['ret_1d'].rolling(20).std()
        
        # Momentum (rendement risque-ajuste)
        df['momentum'] = df['ret_20d'] / (df['vol_20d'] + 1e-8)
        
        # RSI (14 jours)
        delta = prices.diff()
        gain = delta.where(delta > 0, 0)
        loss = -delta.where(delta < 0, 0)
        avg_gain = gain.rolling(14).mean()
        avg_loss = loss.rolling(14).mean()
        rs = avg_gain / (avg_loss + 1e-10)
        df['rsi'] = 100 - (100 / (1 + rs))
        
        # Target : rendement futur risque-ajuste (forward 5 jours)
        future_return = prices.shift(-PRED_LEN) / prices - 1
        future_vol = df['ret_1d'].rolling(PRED_LEN).std().shift(-PRED_LEN)
        df['target_return'] = future_return
        df['target_direction'] = (future_return > 0).astype(int)
        df['target_risk_adj'] = future_return / (future_vol + 1e-8)
        
        # Ajouter les features macro
        for col in macro_df.columns:
            df[col] = macro_df[col]
        
        # Nettoyer
        df = df.ffill().bfill()
        
        # Supprimer les lignes avec des valeurs infinies
        df = df.replace([np.inf, -np.inf], np.nan)
        df = df.ffill().bfill()
        
        all_features[ticker] = df
    
    # Noms des colonnes de features (exclure targets)
    feature_cols = [c for c in all_features[valid_tickers[0]].columns 
                   if not c.startswith('target')]
    
    print(f"Features calculees pour {len(all_features)} actions")
    print(f"Nombre de features par action: {len(feature_cols)}")
    print(f"Features: {feature_cols}")
    
    return all_features, feature_cols

# Calculer les features
all_features, feature_cols = compute_features(close_all, TICKERS_30, MACRO_TICKERS)

print(f"\nExemple pour AAPL:")
print(all_features['AAPL'][feature_cols].describe().round(4))

Features calculees pour 30 actions
Nombre de features par action: 12
Features: ['close', 'ret_1d', 'ret_5d', 'ret_20d', 'vol_10d', 'vol_20d', 'momentum', 'rsi', 'vix_level', 'vix_change', 'yield_spread', 'yield_curve_slope']

Exemple pour AAPL:
           close     ret_1d     ret_5d    ret_20d    vol_10d    vol_20d  \
count  2768.0000  2768.0000  2768.0000  2768.0000  2768.0000  2768.0000   
mean     87.0794     0.0011     0.0055     0.0219     0.0155     0.0160   
std      65.8283     0.0176     0.0370     0.0767     0.0085     0.0075   
min      15.4874    -0.1286    -0.1753    -0.2677     0.0036     0.0048   
25%      28.0757    -0.0070    -0.0159    -0.0243     0.0099     0.0110   
50%      51.1730     0.0010     0.0067     0.0231     0.0137     0.0144   
75%     145.9233     0.0101     0.0272     0.0753     0.0189     0.0191   
max     257.6127     0.1198     0.1841     0.3453     0.0801     0.0680   

        momentum        rsi  vix_level  vix_change  yield_spread  \
count  2768

### Interpretation : Feature engineering

Le feature engineering produit 14 features par echantillon (10 par action + 4 macro).

| Feature | Type | Formule | Utilite |
|---------|------|---------|--------|
| `ret_1d` | Action | `pct_change(1)` | Signal a court terme |
| `ret_5d` | Action | `pct_change(5)` | Tendance hebdomadaire |
| `ret_20d` | Action | `pct_change(20)` | Tendance mensuelle |
| `vol_10d` | Action | `rolling(10).std()` | Volatilite a court terme |
| `vol_20d` | Action | `rolling(20).std()` | Volatilite a moyen terme |
| `momentum` | Action | `ret_20d / vol_20d` | Rendement risque-ajuste |
| `rsi` | Action | RSI(14) | Surachat/survente |
| `vix_level` | Macro | VIX indice | Sentiment peur du marche |
| `vix_change` | Macro | VIX variation | Acceleration de la peur |
| `yield_spread` | Macro | 10Y - 30Y | Pente de la courbe des taux |
| `yield_curve_slope` | Macro | Diff 20j du 10Y | Anticipations de politique monetaire |

**Points cles** :
1. Les features macro sont identiques pour toutes les actions a un jour donne, mais varient dans le temps
2. Le Transformer peut apprendre via l'attention comment le VIX influence differemment chaque secteur
3. Les rendements futurs (`target_return`) servent de cible de regression, `target_direction` de classification

---

## Section 4 : Split temporel strict et normalisation

### Principe du walk-forward

Pour les series temporelles financieres, le split temporel strict est obligatoire :

```
2014 ------[70% Train]------[15% Val]--[15% Test]-- 2025
                             ^          ^
                             |          |
                        Optimisation  Evaluation finale
                        hyperparametres
```

Le StandardScaler est fitte uniquement sur le set d'entrainement pour eviter le data leakage.

In [4]:
# Construction des sequences multi-actifs

class TimeSeriesDataset(Dataset):
    """
    Dataset PyTorch pour sequences temporelles multi-actifs.
    
    Chaque echantillon contient SEQ_LEN jours de features
    pour une action donnee, avec les targets associes.
    """
    
    def __init__(self, sequences, targets_reg, targets_cls):
        self.sequences = torch.FloatTensor(sequences)
        self.targets_reg = torch.FloatTensor(targets_reg)
        self.targets_cls = torch.FloatTensor(targets_cls)
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return (
            self.sequences[idx],
            self.targets_reg[idx],
            self.targets_cls[idx]
        )


def build_sequences(all_features, feature_cols, seq_len=60, pred_len=5):
    """
    Construit les sequences d'entraiement a partir des features.
    
    Concatene les donnees de toutes les actions en un seul
    jeu de sequences, en preservant l'ordre temporel.
    """
    all_X, all_y_reg, all_y_cls = [], [], []
    
    for ticker, df in all_features.items():
        features = df[feature_cols].values
        target_ret = df['target_risk_adj'].values
        target_dir = df['target_direction'].values
        
        # Creer les sequences avec sliding window
        for i in range(seq_len, len(features) - pred_len):
            all_X.append(features[i - seq_len:i])
            all_y_reg.append(target_ret[i])
            all_y_cls.append(target_dir[i])
    
    return (
        np.array(all_X, dtype=np.float32),
        np.array(all_y_reg, dtype=np.float32),
        np.array(all_y_cls, dtype=np.float32)
    )


# Construire les sequences
print("Construction des sequences...")
X_all, y_reg_all, y_cls_all = build_sequences(
    all_features, feature_cols, SEQ_LEN, PRED_LEN
)

print(f"Sequences totales: {len(X_all)}")
print(f"Shape X: {X_all.shape}  [n_samples, seq_len, n_features]")
print(f"Shape y_reg: {y_reg_all.shape}")
print(f"Shape y_cls: {y_cls_all.shape}")
print(f"Distribution direction: {y_cls_all.mean():.2%} up, {1-y_cls_all.mean():.2%} down")

# Split temporel strict (les sequences sont deja ordonnees par date)
n_total = len(X_all)
train_end = int(n_total * TRAIN_RATIO)
val_end = int(n_total * (TRAIN_RATIO + VAL_RATIO))

X_train, y_reg_train, y_cls_train = X_all[:train_end], y_reg_all[:train_end], y_cls_all[:train_end]
X_val, y_reg_val, y_cls_val = X_all[train_end:val_end], y_reg_all[train_end:val_end], y_cls_all[train_end:val_end]
X_test, y_reg_test, y_cls_test = X_all[val_end:], y_reg_all[val_end:], y_cls_all[val_end:]

print(f"\nSplit temporel:")
print(f"  Train: {len(X_train)} samples ({TRAIN_RATIO:.0%})")
print(f"  Val:   {len(X_val)} samples ({VAL_RATIO:.0%})")
print(f"  Test:  {len(X_test)} samples ({1-TRAIN_RATIO-VAL_RATIO:.0%})")

# Normalisation : fit sur train uniquement
n_features = X_train.shape[2]
scaler = StandardScaler()
X_train_2d = X_train.reshape(-1, n_features)
scaler.fit(X_train_2d)

X_train = scaler.transform(X_train_2d).reshape(X_train.shape)
X_val = scaler.transform(X_val.reshape(-1, n_features)).reshape(X_val.shape)
X_test = scaler.transform(X_test.reshape(-1, n_features)).reshape(X_test.shape)

print(f"\nNormalisation appliquee (StandardScaler fitte sur train)")
print(f"  Features: {n_features}")
print(f"  Train mean: {X_train.mean():.4f}, std: {X_train.std():.4f}")

# Creer les DataLoaders
train_dataset = TimeSeriesDataset(X_train, y_reg_train, y_cls_train)
val_dataset = TimeSeriesDataset(X_val, y_reg_val, y_cls_val)
test_dataset = TimeSeriesDataset(X_test, y_reg_test, y_cls_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nDataLoaders crees:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches:   {len(val_loader)}")
print(f"  Test batches:  {len(test_loader)}")

Construction des sequences...


Sequences totales: 81090
Shape X: (81090, 60, 12)  [n_samples, seq_len, n_features]
Shape y_reg: (81090,)
Shape y_cls: (81090,)
Distribution direction: 56.35% up, 43.65% down

Split temporel:
  Train: 56763 samples (70%)
  Val:   12163 samples (15%)
  Test:  12164 samples (15%)



Normalisation appliquee (StandardScaler fitte sur train)
  Features: 12


  Train mean: 0.0000, std: 1.0000

DataLoaders crees:
  Train batches: 1774
  Val batches:   381
  Test batches:  381


### Interpretation : Split et normalisation

La preparation des donnees suit un protocole strict pour eviter le data leakage :

| Etape | Detail |
|-------|--------|
| **Construction** | Sliding window de 60 jours sur chaque action, concatenees |
| **Split** | Temporel strict (70/15/15), jamais de shuffle entre splits |
| **Normalisation** | StandardScaler fitte uniquement sur le train set |

**Points cles** :
1. Le split temporel est essentiel en finance : on ne peut pas utiliser des donnees futures pour entrainer
2. Le scaler est fitte sur le train uniquement et applique (transform) sur val et test
3. La distribution des targets (up/down) est proche de 50/50, ce qui est attendu sur un marche efficient
4. Le DataLoader shuffle les batchs d'entrainement mais preserve l'ordre en validation/test

---

## Section 5 : Modele Transformer Encoder

### Positional Encoding et auto-attention

Le Transformer Encoder utilise deux composants cles :

1. **Positional Encoding** : Les Transformers n'ont pas de notion d'ordre natif. L'encodage positionnel sinusooidal injecte l'information temporelle :

$$PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

2. **Auto-attention multi-tete** : Chaque tete apprend une projection differente de la sequence, permettant au modele de capturer simultanement des tendances a differentes echelles.

### Dual-head output

Le modele produit deux sorties (meme pattern que QC-Py-30) :
- **Regression head** : Prediction du rendement futur risque-ajuste
- **Classification head** : Prediction de la direction (hausse/baisse)

In [5]:
class PositionalEncoding(nn.Module):
    """
    Encodage positionnel sinusooidal standard du Transformer.
    
    Injecte l'information d'ordre temporel dans les embeddings,
    car le Transformer n'a pas de notion d'ordre natif.
    
    Reference : "Attention Is All You Need" (Vaswani et al., 2017)
    """
    
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Calculer les encodages positionnels une seule fois
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        
        pe[:, 0::2] = torch.sin(position * div_term)  # Indices pairs
        pe[:, 1::2] = torch.cos(position * div_term)  # Indices impairs
        
        # Enregistrer comme buffer (pas de gradient)
        self.register_buffer('pe', pe.unsqueeze(0))  # [1, max_len, d_model]
    
    def forward(self, x):
        """
        Parameters
        ----------
        x : Tensor [batch, seq_len, d_model]
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class TransformerPredictor(nn.Module):
    """
    Transformer Encoder pour prediction de series temporelles financieres.
    
    Architecture :
    - Input projection (features -> d_model)
    - Positional encoding (sinusoidal)
    - TransformerEncoder (N couches)
    - Global average pooling
    - Dual head : regression + classification
    
    Parameters
    ----------
    input_dim : int
        Nombre de features d'entree
    d_model : int
        Dimension interne du Transformer
    nhead : int
        Nombre de tetes d'attention
    num_layers : int
        Nombre de couches encodeur
    dim_feedforward : int
        Dimension du reseau feed-forward
    dropout : float
        Taux de dropout
    """
    
    def __init__(
        self,
        input_dim,
        d_model=256,
        nhead=8,
        num_layers=4,
        dim_feedforward=1024,
        dropout=0.1
    ):
        super().__init__()
        
        self.d_model = d_model
        
        # Projection d'entree
        self.input_proj = nn.Linear(input_dim, d_model)
        
        # Encodage positionnel
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        # Couches de l'encodeur Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation='gelu'
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            enable_nested_tensor=False
        )
        
        # Tete de regression (rendement futur risque-ajuste)
        self.reg_head = nn.Sequential(
            nn.Linear(d_model, d_model // 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 4, 1)
        )
        
        # Tete de classification (direction hausse/baisse)
        self.cls_head = nn.Sequential(
            nn.Linear(d_model, d_model // 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 4, 1)
        )
    
    def forward(self, x, return_attention=False):
        """
        Forward pass.
        
        Parameters
        ----------
        x : Tensor [batch, seq_len, input_dim]
        return_attention : bool
            Si True, retourne aussi les poids d'attention de la derniere couche
        
        Returns
        -------
        reg_output : Tensor [batch, 1]
        cls_output : Tensor [batch, 1]
        attention_weights : Tensor [batch, nhead, seq_len, seq_len] (optionnel)
        """
        # Projection d'entree
        x = self.input_proj(x)  # [batch, seq_len, d_model]
        
        # Encodage positionnel
        x = self.pos_encoder(x)
        
        # Transformer encodeur
        if return_attention:
            # Extraire les poids d'attention de la derniere couche
            x_encoded = x
            attention_weights = None
            for layer in self.transformer_encoder.layers:
                # Utiliser self_attn manuellement pour extraire les poids
                src2, attn = layer.self_attn(
                    x_encoded, x_encoded, x_encoded,
                    need_weights=True,
                    average_attn_weights=False
                )
                attention_weights = attn
                x_encoded = layer.norm1(x_encoded + layer.dropout1(src2))
                # FFN
                src2 = layer.linear2(layer.dropout(layer.activation(layer.linear1(x_encoded))))
                x_encoded = layer.norm2(x_encoded + layer.dropout2(src2))
            x = x_encoded
        else:
            x = self.transformer_encoder(x)  # [batch, seq_len, d_model]
        
        # Global average pooling sur la dimension temporelle
        x = x.mean(dim=1)  # [batch, d_model]
        
        # Sorties
        reg_output = self.reg_head(x)  # [batch, 1]
        cls_output = self.cls_head(x)  # [batch, 1]
        
        if return_attention:
            return reg_output, cls_output, attention_weights
        return reg_output, cls_output


# Instancier le modele
model = TransformerPredictor(
    input_dim=n_features,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT
).to(device)

# Resume du modele
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 60)
print("TRANSFORMER ENCODER - ARCHITECTURE")
print("=" * 60)
print(model)
print(f"\nParametres totaux: {n_params:,}")
print(f"Parametres entrainables: {n_trainable:,}")
print(f"\nTaille estimee en memoire: {n_params * 4 / 1024 / 1024:.1f} MB (float32)")

TRANSFORMER ENCODER - ARCHITECTURE
TransformerPredictor(
  (input_proj): Linear(in_features=12, out_features=256, bias=True)
  (pos_encoder): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (reg_head): Sequential(
    (0): Linear(in_features=256, out_feature

### Interpretation : Architecture du Transformer

Le modele TransformerPredictor est construit avec les composants PyTorch natifs.

| Composant | Dimension | Role |
|-----------|-----------|------|
| `input_proj` | 14 -> 256 | Projection des features vers l'espace latent |
| `pos_encoder` | sinusoidal | Injection de l'information temporelle |
| `TransformerEncoder` | 4 couches x 8 tetes | Auto-attention + FFN (dim=1024) |
| `reg_head` | 256 -> 64 -> 1 | Prediction du rendement risque-ajuste |
| `cls_head` | 256 -> 64 -> 1 | Prediction de direction (logit) |

**Points cles** :
1. L'encodage positionnel sinusooidal permet au modele de differencier les positions dans le temps
2. L'activation GELU dans les tetes de sortie est plus douce que ReLU et mieux adaptee aux Transformers
3. Le `batch_first=True` simplifie la gestion des dimensions `[batch, seq_len, d_model]`
4. Le nombre de parametres (~1-2M) est significativement plus eleve que le LSTM mais reste raisonnable
5. L'extraction des poids d'attention se fait en desactivant le forward standard et en appelant `self_attn` manuellement

---

## Section 6 : Entrainement GPU

### Strategie d'entrainement

L'entrainement combine plusieurs techniques pour stabiliser la convergence :

| Technique | Parametre | Objectif |
|-----------|-----------|----------|
| **AdamW** | lr=0.0005, weight_decay=0.01 | Optimiseur avec decouplage poids/decay |
| **CosineAnnealingLR** | T_max=EPOCHS | Scheduler cosinus pour descente progressive du LR |
| **HuberLoss + BCELoss** | ratio 0.5 | Loss combinee pour regression + classification |
| **Gradient clipping** | max_norm=1.0 | Eviter l'explosion du gradient dans les Transformers |
| **Early stopping** | patience=7 | Arreter avant overfitting |
| **torch.cuda.empty_cache()** | Entre epochs | Liberer la memoire GPU pour la securite thermique |

In [6]:
# Fonctions d'entrainement et d'evaluation avec AMP (Mixed Precision)
# AMP reduit la consommation VRAM et la chaleur GPU de ~50%

def train_epoch(model, loader, optimizer, scheduler, device, grad_scaler=None, thermal_fn=None):
    """
    Entraîne le modele pour une epoch avec support AMP optionnel.
    
    Parameters
    ----------
    grad_scaler : torch.amp.GradScaler or None
        Si fourni, utilise AMP (mixed precision) pour reduire VRAM/chauffe.
    thermal_fn : callable or None
        Fonction de check thermique intra-epoch (batch_idx) -> temp.
    """
    model.train()
    total_loss = 0
    total_reg_loss = 0
    total_cls_loss = 0
    correct = 0
    total = 0
    
    use_amp = grad_scaler is not None and grad_scaler.is_enabled()
    reg_criterion = nn.HuberLoss()
    cls_criterion = nn.BCEWithLogitsLoss()
    
    for batch_idx, (x_batch, y_reg_batch, y_cls_batch) in enumerate(loader):
        # Intra-epoch thermal check (every N batches)
        if thermal_fn is not None:
            thermal_fn(batch_idx)
        
        x_batch = x_batch.to(device)
        y_reg_batch = y_reg_batch.to(device)
        y_cls_batch = y_cls_batch.to(device)
        
        optimizer.zero_grad()
        
        # Forward avec AMP
        with torch.amp.autocast('cuda', enabled=use_amp):
            reg_out, cls_out = model(x_batch)
            reg_loss = reg_criterion(reg_out.squeeze(), y_reg_batch)
            cls_loss = cls_criterion(cls_out.squeeze(), y_cls_batch)
            loss = reg_loss + 0.5 * cls_loss
        
        # Backward avec AMP
        if use_amp:
            grad_scaler.scale(loss).backward()
            grad_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            grad_scaler.step(optimizer)
            grad_scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        # Metriques
        total_loss += loss.item() * len(y_reg_batch)
        total_reg_loss += reg_loss.item() * len(y_reg_batch)
        total_cls_loss += cls_loss.item() * len(y_reg_batch)
        
        # Direction accuracy
        preds = (torch.sigmoid(cls_out.squeeze()).float() > 0.5).float()
        correct += (preds == y_cls_batch).sum().item()
        total += len(y_cls_batch)
    
    scheduler.step()
    
    return {
        'loss': total_loss / total,
        'reg_loss': total_reg_loss / total,
        'cls_loss': total_cls_loss / total,
        'dir_acc': correct / total
    }


def evaluate(model, loader, device):
    """Evalue le modele sur un DataLoader avec AMP pour la coherence."""
    model.eval()
    total_loss = 0
    total_reg_loss = 0
    total_cls_loss = 0
    correct = 0
    total = 0
    
    reg_criterion = nn.HuberLoss()
    cls_criterion = nn.BCEWithLogitsLoss()
    
    all_preds_reg = []
    all_targets_reg = []
    all_preds_cls = []
    all_targets_cls = []
    
    use_amp = torch.cuda.is_available()
    
    with torch.no_grad():
        for x_batch, y_reg_batch, y_cls_batch in loader:
            x_batch = x_batch.to(device)
            y_reg_batch = y_reg_batch.to(device)
            y_cls_batch = y_cls_batch.to(device)
            
            with torch.amp.autocast('cuda', enabled=use_amp):
                reg_out, cls_out = model(x_batch)
            
            # Calculer la loss en float32 pour stabilite numerique
            reg_out_f = reg_out.squeeze().float()
            cls_out_f = cls_out.squeeze().float()
            reg_loss = reg_criterion(reg_out_f, y_reg_batch)
            cls_loss = cls_criterion(cls_out_f, y_cls_batch)
            loss = reg_loss + 0.5 * cls_loss
            
            total_loss += loss.item() * len(y_reg_batch)
            total_reg_loss += reg_loss.item() * len(y_reg_batch)
            total_cls_loss += cls_loss.item() * len(y_reg_batch)
            
            preds = (torch.sigmoid(cls_out_f) > 0.5).float()
            correct += (preds == y_cls_batch).sum().item()
            total += len(y_cls_batch)
            
            all_preds_reg.append(reg_out_f.cpu().numpy())
            all_targets_reg.append(y_reg_batch.cpu().numpy())
            all_preds_cls.append(preds.cpu().numpy())
            all_targets_cls.append(y_cls_batch.cpu().numpy())
    
    return {
        'loss': total_loss / total,
        'reg_loss': total_reg_loss / total,
        'cls_loss': total_cls_loss / total,
        'dir_acc': correct / total,
        'preds_reg': np.concatenate(all_preds_reg),
        'targets_reg': np.concatenate(all_targets_reg),
        'preds_cls': np.concatenate(all_preds_cls),
        'targets_cls': np.concatenate(all_targets_cls)
    }

print("Fonctions d'entrainement et d'evaluation definies (avec AMP + thermal checks)")


Fonctions d'entrainement et d'evaluation definies (avec AMP)


### Preparation de la boucle d'entrainement

Les fonctions `train_epoch` et `evaluate` implementent :

- **Loss combinee** : HuberLoss (robuste aux outliers pour les rendements) + 0.5 * BCELoss (classification direction)
- **Gradient clipping** : Limite la norme des gradients a 1.0 pour stabiliser l'entrainement
- **Metrics tracking** : Loss totale, loss regression, loss classification, direction accuracy

Le ratio 0.5 entre regression et classification equilibre les deux objectifs sans que l'un domine l'autre.

In [7]:
# Boucle d'entrainement avec checkpoint, thermal watchdog et AMP
# Utilise le module shared/gpu_training.py pour eviter de dupliquer le code

import sys
import importlib

# Resolve shared module: works from notebook dir, Papermill, and CLI
_shared_candidates = [
    os.path.abspath(os.path.join(os.getcwd(), '..')),                          # Running from Python/ dir
    os.path.abspath(os.path.join(os.getcwd(), 'MyIA.AI.Notebooks', 'QuantConnect')),  # Running from repo root
    os.path.abspath(os.path.join(os.getcwd(), 'QuantConnect')),                # Running from notebooks root
]
for _p in _shared_candidates:
    if os.path.isfile(os.path.join(_p, 'shared', 'gpu_training.py')):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        break

from shared.gpu_training import TrainingCheckpoint, setup_amp, get_gpu_temp

# --- Notebook directory (robust across Jupyter / Papermill / CLI) ---
_nb_dir = os.path.abspath(os.path.join(os.getcwd(), 'MyIA.AI.Notebooks', 'QuantConnect', 'Python'))
if not os.path.isfile(os.path.join(_nb_dir, 'QC-Py-31-Transformer-Training.ipynb')):
    _nb_dir = os.getcwd()  # Already in notebook dir (Jupyter)

# --- Configuration checkpoint ---
checkpoint_path = os.path.join(_nb_dir, 'transformer_checkpoint.pt')
model_save_path = os.path.join(_nb_dir, 'transformer_multiasset_model.pt')

# --- AMP (Mixed Precision) ---
use_amp, grad_scaler = setup_amp()
print(f"AMP: {'Active (GPU)' if use_amp else 'Desactive (CPU)'}")

# --- Optimiseur et scheduler ---
optimizer = optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.01
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=LR * 0.01
)

# --- Historique ---
history = {
    'train_loss': [], 'val_loss': [],
    'train_reg_loss': [], 'val_reg_loss': [],
    'train_cls_loss': [], 'val_cls_loss': [],
    'train_dir_acc': [], 'val_dir_acc': []
}

# --- Resume checkpoint (3 cas: modele final / checkpoint / from scratch) ---
ckpt_manager = TrainingCheckpoint(
    checkpoint_path=checkpoint_path,
    model_save_path=model_save_path,
    max_temp=80,
    thermal_check_every=10,
    cool_sleep=15,
    patience=7
)
start_epoch, history = ckpt_manager.resume(
    model, optimizer, scheduler, grad_scaler,
    device=device, default_history=history
)

# --- Boucle d'entrainement ---
if start_epoch >= 0 and start_epoch < EPOCHS:
    print("=" * 70)
    print("ENTRAINEMENT TRANSFORMER ENCODER (AMP + Thermal Watchdog)")
    print("=" * 70)
    print(f"Device: {device}")
    print(f"Epochs: {start_epoch+1}-{EPOCHS}, Batch: {BATCH_SIZE}, LR: {LR}")
    print(f"Thermal limit: {ckpt_manager.max_temp}C")
    print()

    for epoch in range(start_epoch, EPOCHS):
        epoch_start = time.time()
        
        # Thermal check via le helper
        ckpt_manager.thermal_check()
        
        # Entrainement avec AMP
        train_metrics = train_epoch(model, train_loader, optimizer, scheduler, device, grad_scaler, thermal_fn=ckpt_manager.batch_thermal_check)
        
        # Validation
        val_metrics = evaluate(model, val_loader, device)
        
        epoch_time = time.time() - epoch_start
        
        # Sauvegarder l'historique
        for key in history:
            prefix = 'train_' if key.startswith('train') else 'val_'
            metric_key = key.replace(prefix, '')
            metrics = train_metrics if prefix == 'train_' else val_metrics
            history[key].append(metrics[metric_key])
        
        # GPU temp pour affichage
        gpu_temp = get_gpu_temp()
        temp_str = f" | GPU: {gpu_temp}C" if gpu_temp > 0 else ""
        
        # Afficher le progres
        print(
            f"Epoch {epoch+1:2d}/{EPOCHS} | "
            f"Train Loss: {train_metrics['loss']:.4f} "
            f"(reg={train_metrics['reg_loss']:.4f}, cls={train_metrics['cls_loss']:.4f}) | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Dir Acc: {val_metrics['dir_acc']:.2%} | "
            f"Time: {epoch_time:.1f}s{temp_str}"
        )
        
        # Checkpoint update
        is_best = ckpt_manager.update(
            epoch, val_metrics['loss'], history,
            model, optimizer, scheduler, grad_scaler
        )
        if is_best:
            print(f"  -> Nouveau meilleur modele (val_loss={val_metrics['loss']:.4f})")
        
        # Early stopping
        if ckpt_manager.should_stop():
            print(f"\nEarly stopping a l'epoch {epoch+1} (patience={ckpt_manager.patience})")
            break
        
        # Liberer memoire GPU
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Sauvegarder le modele final
    ckpt_manager.finalize(model, extra={
        'scaler': scaler,
        'config': {
            'input_dim': n_features,
            'd_model': D_MODEL, 'nhead': NHEAD,
            'num_layers': NUM_LAYERS,
            'dim_feedforward': DIM_FEEDFORWARD,
            'dropout': DROPOUT,
            'seq_len': SEQ_LEN, 'pred_len': PRED_LEN,
            'feature_cols': feature_cols,
            'tickers': TICKERS_30,
        }
    })
    
    print(f"\nEntrainement termine.")
    print(f"Meilleure val_loss: {ckpt_manager.best_val_loss:.4f}")
    print(f"Checkpoint de reprise: {checkpoint_path} (ne pas commit)")
else:
    print("\nModele deja entraine et charge. Aucun entrainement necessaire.")

AMP: Active (GPU)


Aucun checkpoint ni modele final. Demarrage from scratch.
ENTRAINEMENT TRANSFORMER ENCODER (AMP + Thermal Watchdog)
Device: cuda
Epochs: 1-25, Batch: 32, LR: 0.0005
Thermal limit: 87C



Epoch  1/25 | Train Loss: 2.0622 (reg=1.7191, cls=0.6862) | Val Loss: 2.0237 | Dir Acc: 57.34% | Time: 118.6s | GPU: 68C
  -> Nouveau meilleur modele (val_loss=2.0237)


Epoch  2/25 | Train Loss: 2.0602 (reg=1.7175, cls=0.6855) | Val Loss: 2.0233 | Dir Acc: 57.34% | Time: 114.1s | GPU: 72C


  -> Nouveau meilleur modele (val_loss=2.0233)


...

Epoch  8/25 | Train Loss: 2.0597 (reg=1.7171, cls=0.6852) | Val Loss: 2.0213 | Dir Acc: 57.34% | Time: 133.6s | GPU: 82C


Epoch  9/25 | Train Loss: 2.0597 (reg=1.7171, cls=0.6854) | Val Loss: 2.0240 | Dir Acc: 57.34% | Time: 131.2s | GPU: 84C


Epoch 10/25 | Train Loss: 2.0596 (reg=1.7170, cls=0.6852) | Val Loss: 2.0216 | Dir Acc: 57.34% | Time: 108.7s | GPU: 87C


Epoch 11/25 | Train Loss: 2.0594 (reg=1.7168, cls=0.6852) | Val Loss: 2.0237 | Dir Acc: 57.34% | Time: 67.7s | GPU: 88C
  [THERMAL] GPU 88C > 87C, pause 15s...


Epoch 12/25 | Train Loss: 2.0597 (reg=1.7171, cls=0.6853) | Val Loss: 2.0218 | Dir Acc: 57.34% | Time: 90.2s | GPU: 89C
  [THERMAL] GPU 89C > 87C, pause 15s...


### Interpretation : Entrainement

La boucle d'entrainement applique les bonnes pratiques pour les Transformers en finance :

| Technique | Effet observe |
|-----------|---------------|
| **CosineAnnealingLR** | Descente progressive du LR, convergence plus stable |
| **Gradient clipping** | Previent l'explosion du gradient dans les couches d'attention |
| **Early stopping** | Arrete l'entrainement avant overfitting (patience=7) |
| **torch.cuda.empty_cache()** | Limite l'accumulation memoire entre epochs |

**Points cles** :
1. La direction accuracy en validation devrait se situer entre 50% et 56% (les marches sont difficiles a predire)
2. La loss devrait decroitre de maniere monotone sur le train set, la val loss peut stagner ou remonter
3. Le temps par epoch depend du GPU ; sur CPU, attendez 2-5x plus lent
4. Le early stopping est essentiel : les Transformers sur-apprennent vite sur les series financieres bruitees

---

## Section 7 : Courbes d'apprentissage

### Visualisation de la convergence

Trois graphiques permettent de diagnostiquer l'entrainement :
1. **Loss totale** : Train vs Validation pour detecter l'overfitting
2. **Decomposition** : Loss regression vs classification
3. **Direction accuracy** : Capacite du modele a predire la direction du marche

In [ ]:
# Visualisation des courbes d'apprentissage
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Loss totale
ax1 = axes[0]
ax1.plot(history['train_loss'], label='Train', color='steelblue', linewidth=1.5)
ax1.plot(history['val_loss'], label='Validation', color='coral', linewidth=1.5)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss (Huber + 0.5*BCE)')
ax1.set_title('Loss Totale', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Decomposition regression / classification
ax2 = axes[1]
ax2.plot(history['train_reg_loss'], label='Train Reg', color='steelblue', linestyle='--')
ax2.plot(history['val_reg_loss'], label='Val Reg', color='coral', linestyle='--')
ax2.plot(history['train_cls_loss'], label='Train Cls', color='steelblue', linestyle=':')
ax2.plot(history['val_cls_loss'], label='Val Cls', color='coral', linestyle=':')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Decomposition Reg/Cls', fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# 3. Direction accuracy
ax3 = axes[2]
ax3.plot(history['train_dir_acc'], label='Train', color='steelblue', linewidth=1.5)
ax3.plot(history['val_dir_acc'], label='Validation', color='coral', linewidth=1.5)
ax3.axhline(0.50, color='gray', linestyle='--', alpha=0.5, label='Random (50%)')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Direction Accuracy')
ax3.set_title('Direction Accuracy', fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_ylim(0.45, 0.65)

plt.tight_layout()
plt.show()

# Resume final de l'entrainement
if history['train_loss']:
    print("Resume de l'entrainement:")
    print(f"  Epochs effectues: {len(history['train_loss'])}")
    print(f"  Train loss finale: {history['train_loss'][-1]:.4f}")
    print(f"  Val loss finale:   {history['val_loss'][-1]:.4f}")
    print(f"  Val dir accuracy:  {history['val_dir_acc'][-1]:.2%}")
else:
    print("Aucun historique d'entrainement (modele pre-entraine charge).")

plt.savefig(os.path.join(_nb_dir if '_nb_dir' in dir() else os.getcwd(), 'training_curves.png'), dpi=100, bbox_inches='tight')

### Interpretation : Courbes d'apprentissage

Les trois graphiques permettent de diagnostiquer la qualite de l'entrainement :

| Graphique | Observation attendue | Signification |
|-----------|---------------------|---------------|
| **Loss totale** | Decroissance puis plateau | Le modele converge |
| **Reg/Cls decomposition** | Reg domine au debut, Cls stabilise | La regression est plus difficile |
| **Direction accuracy** | Au-dessus de 50% | Le modele a un edge predictif |

**Points cles** :
1. Un ecart croissant entre train et val loss indique de l'overfitting
2. La direction accuracy superieure a 50% est un signal positif, meme si l'ecart est faible
3. La loss de classification (BCE) converge generalement plus vite que la loss de regression
4. En finance, une direction accuracy de 53-55% est deja exploitable avec un bon money management

---

## Section 8 : Evaluation sur le set de test

L'evaluation finale se fait sur le set de test (15% le plus recent), qui n'a jamais ete vu pendant l'entrainement.

In [ ]:
# Evaluation sur le set de test
test_metrics = evaluate(model, test_loader, device)

print("=" * 60)
print("EVALUATION SUR LE SET DE TEST")
print("=" * 60)

# Metriques de regression
preds_reg = test_metrics['preds_reg']
targets_reg = test_metrics['targets_reg']
mse = mean_squared_error(targets_reg, preds_reg)
mae = mean_absolute_error(targets_reg, preds_reg)
correlation = np.corrcoef(preds_reg, targets_reg)[0, 1]

# Metriques de classification
dir_acc = test_metrics['dir_acc']

print(f"\nMetriques de regression:")
print(f"  MSE:          {mse:.6f}")
print(f"  MAE:          {mae:.6f}")
print(f"  Correlation:  {correlation:.4f}")

print(f"\nMetriques de classification:")
print(f"  Direction Accuracy: {dir_acc:.2%}")
print(f"  Amelioration vs random: {(dir_acc - 0.5) * 100:.1f} points de %")

# Analyse par quantile de prediction (robuste aux doublons)
preds_reg_df = pd.DataFrame({'pred': preds_reg, 'target': targets_reg})
try:
    _binned = pd.qcut(preds_reg_df['pred'], 5, duplicates='drop')
    _n_actual = _binned.nunique()
    _labels = [f'Q{i+1}' for i in range(_n_actual)]
    preds_reg_df['pred_quantile'] = pd.qcut(preds_reg_df['pred'], _n_actual, labels=_labels, duplicates='drop')
    quantile_analysis = preds_reg_df.groupby('pred_quantile')['target'].mean()
    print(f"\nRendement moyen par quantile de prediction ({_n_actual} quantiles):")
    print(quantile_analysis.round(4))
except ValueError:
    print("\nPas assez de variance dans les predictions pour une analyse par quantile.")

### Interpretation : Evaluation test

Les metriques sur le set de test mesurent la capacite de generalisation du modele.

| Metrique | Valeur typique | Signification |
|----------|----------------|---------------|
| **Correlation** | 0.02-0.08 | Faible mais positive, attendue sur marche liquide |
| **Direction Accuracy** | 51-55% | Modeste mais potentiellement exploitable |
| **MSE/MAE** | Dependent de l'echelle | Comparaison relative avec le LSTM |

**Points cles** :
1. L'analyse par quantile est cruciale : si le rendement moyen croit monotone de Q1 a Q5, le modele classe correctement les actions
2. Une correlation positive meme faible (0.03-0.05) peut generer des profits si combinee avec une bonne gestion du risque
3. En pratique, la direction accuracy et la correlation sont plus importantes que le MSE pour le trading
4. Les resultats sur donnees de marche reelles sont toujours plus modestes que sur des donnees synthetiques

---

## Section 9 : Visualisation des predictions

Le scatter plot predictions vs realite permet de verifier visuellement la qualite de la regression.

In [ ]:
# Visualisation predictions vs realite
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot predictions vs cibles
ax1 = axes[0]
# Sous-echantillonner pour la lisibilite
n_show = min(2000, len(preds_reg))
idx = np.random.choice(len(preds_reg), n_show, replace=False)
idx = np.sort(idx)

ax1.scatter(
    targets_reg[idx], preds_reg[idx],
    alpha=0.3, s=8, color='steelblue'
)
# Ligne diagonale parfaite
lims = [
    min(targets_reg[idx].min(), preds_reg[idx].min()),
    max(targets_reg[idx].max(), preds_reg[idx].max())
]
ax1.plot(lims, lims, 'r--', alpha=0.5, label='Prediction parfaite')
ax1.set_xlabel('Rendement reel (risk-adj)')
ax1.set_ylabel('Rendement predit')
ax1.set_title('Predictions vs Realite', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Distribution des predictions par direction
ax2 = axes[1]
preds_cls = test_metrics['preds_cls']
targets_cls = test_metrics['targets_cls']

# Calculer la probabilite predite (via le logit)
correct_mask = preds_cls == targets_cls
labels = ['Incorrect', 'Correct']
sizes = [len(correct_mask) - correct_mask.sum(), correct_mask.sum()]
colors = ['coral', 'seagreen']

ax2.bar(labels, sizes, color=colors, edgecolor='black')
ax2.set_ylabel('Nombre de predictions')
ax2.set_title(f'Direction Accuracy: {dir_acc:.2%}', fontweight='bold')
for i, (label, size) in enumerate(zip(labels, sizes)):
    ax2.text(i, size + 100, f'{size:,}', ha='center', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### Interpretation : Visualisation des predictions

Les deux graphiques offrent une vue qualitative des predictions du modele :

| Graphique | Observation attendue |
|-----------|---------------------|
| **Scatter plot** | Nuage de pointscentre sur la diagonale, avec dispersion importante |
| **Bar chart direction** | Leger avantage des predictions correctes sur les incorrectes |

**Points cles** :
1. Le scatter plot ne montre PAS une correlation lineaire forte -- c'est normal en finance
2. Ce qui compte est la capacite du modele a **classer** les actions (top decile vs bottom decile)
3. La direction accuracy est la metrique la plus actionnable pour le trading
4. L'asymetrie des erreurs (erreur sur les baisses vs erreur sur les hausse) peut indiquer un biais du modele

---

## Section 10 : Visualisation de l'attention

L'un des avantages du Transformer est l'interpretabilite des poids d'attention. Nous extrayons les poids de la derniere couche et les visualisons sous forme de heatmap.

In [ ]:
# Extraction et visualisation des poids d'attention

model.eval()

# Prendre un echantillon du set de test
sample_x = torch.FloatTensor(X_test[:1]).to(device)

# Forward pass avec extraction des poids d'attention
with torch.no_grad():
    reg_out, cls_out, attn_weights = model(sample_x, return_attention=True)

# attn_weights shape: [batch, nhead, seq_len, seq_len]
print(f"Forme des poids d'attention: {attn_weights.shape}")
print(f"  batch=1, nhead={attn_weights.shape[1]}, seq_len={attn_weights.shape[2]}")

# Visualiser la heatmap d'attention (moyenne sur les tetes)
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Attention moyenne sur toutes les tetes
avg_attn = attn_weights[0].mean(dim=0).cpu().numpy()  # [seq_len, seq_len]

ax1 = axes[0, 0]
im1 = ax1.imshow(avg_attn, aspect='auto', cmap='YlOrRd', interpolation='nearest')
ax1.set_title('Attention Moyenne (toutes tetes)', fontweight='bold')
ax1.set_xlabel('Position cle (key)')
ax1.set_ylabel('Position requete (query)')
plt.colorbar(im1, ax=ax1, label='Poids')

# 2-4. Attention de 3 tetes individuelles
heads_to_show = [0, 3, 7]
for idx, (ax, head_idx) in enumerate(zip(
    [axes[0, 1], axes[1, 0], axes[1, 1]],
    heads_to_show
)):
    head_attn = attn_weights[0, head_idx].cpu().numpy()
    im = ax.imshow(head_attn, aspect='auto', cmap='YlOrRd', interpolation='nearest')
    ax.set_title(f'Tete {head_idx} d\'attention', fontweight='bold')
    ax.set_xlabel('Position cle (key)')
    ax.set_ylabel('Position requete (query)')
    plt.colorbar(im, ax=ax, label='Poids')

plt.tight_layout()
plt.show()

# Analyser la distribution des poids d'attention
print("\nAnalyse des poids d'attention:")
for head_idx in heads_to_show:
    head_attn = attn_weights[0, head_idx].cpu().numpy()
    # Quelle position recoit le plus d'attention en moyenne ?
    avg_col = head_attn.mean(axis=0)
    top_positions = np.argsort(avg_col)[-5:][::-1]
    print(f"  Tete {head_idx} - Top 5 positions les plus attentives: {top_positions}")
    print(f"    Poids correspondants: {avg_col[top_positions].round(3)}")

### Interpretation : Poids d'attention

La visualisation de l'attention revele quelles periodes historiques le modele considere comme les plus pertinentes pour sa prediction.

| Observation | Signification |
|-------------|---------------|
| **Diagonale dominante** | Chaque position s'attend surtout a elle-meme (pattern local) |
| **Bande horizontale en bas** | Les positions recentes attendent les positions anciennes (memoire longue) |
| **Tetes differentes** | Chaque tete capture un pattern different (court terme, long terme, points de retournement) |

**Points cles** :
1. L'attention est un mecanisme **adaptable** : le modele apprend quelles positions regarder, pas de regle fixe
2. Les tetes qui se concentrent sur les positions recentes capturent probablement la dynamique a court terme
3. Les tetes qui regardent des positions eloignees capturent les tendances de fond
4. En finance, on s'attend a ce que les evenements de volatilite recente (crash, rally) attirent plus d'attention

---

## Section 11 : Backtest simplifie du signal

### Strategie de trading simplifiee

Nous appliquons un backtest simplifie : a chaque date de rebalancement, le modele predit le rendement futur de chaque action, et on investit dans le top decile.

In [ ]:
# Backtest simplifie du signal Transformer

def backtest_transformer_signal(
    preds_reg, targets_reg, targets_cls,
    n_bins=5, rebalance_freq=20
):
    """
    Backtest simplifie : long le top quintile des predictions.
    
    A chaque pas de rebalancement :
    1. Trier les actions par prediction de rendement
    2. Acheter le top quintile (short le bottom si souhaite)
    3. Calculer le rendement du portefeuille
    """
    n_samples = len(preds_reg)
    portfolio_returns = []
    dates_indices = []
    
    for i in range(0, n_samples - rebalance_freq, rebalance_freq):
        # Prendre les predictions de cette periode
        batch_preds = preds_reg[i:i+rebalance_freq]
        batch_targets = targets_reg[i:i+rebalance_freq]
        
        # Trier par prediction (meilleur rendement attendu)
        sorted_indices = np.argsort(batch_preds)
        
        # Top quintile
        top_n = max(1, len(batch_preds) // n_bins)
        top_indices = sorted_indices[-top_n:]
        
        # Rendement realise du portefeuille
        port_return = batch_targets[top_indices].mean()
        portfolio_returns.append(port_return)
        dates_indices.append(i)
    
    return np.array(portfolio_returns), dates_indices


# Executer le backtest
port_returns, port_dates = backtest_transformer_signal(
    preds_reg, targets_reg, test_metrics['targets_cls']
)

# Calculer les metriques de performance
cumulative = np.cumprod(1 + port_returns)
total_return = cumulative[-1] - 1
annualized_return = (1 + total_return) ** (252 / max(len(port_returns) * 20, 1)) - 1
vol = np.std(port_returns) * np.sqrt(252 / 20)
sharpe = annualized_return / vol if vol > 0 else 0

# Max drawdown
running_max = np.maximum.accumulate(cumulative)
drawdown = (cumulative - running_max) / running_max
max_dd = drawdown.min()

# Benchmark : moyenne de toutes les actions (equal weight)
benchmark_returns = []
for i in range(0, len(targets_reg) - 20, 20):
    benchmark_returns.append(targets_reg[i:i+20].mean())
benchmark_cumulative = np.cumprod(1 + np.array(benchmark_returns))

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Rendement cumule
ax1 = axes[0]
ax1.plot(cumulative, label='Transformer (Top Quintile)', color='steelblue', linewidth=1.5)
ax1.plot(benchmark_cumulative, label='Benchmark (Equal Weight)', color='gray', linewidth=1, alpha=0.7)
ax1.set_xlabel('Periodes de rebalancement')
ax1.set_ylabel('Rendement cumule')
ax1.set_title('Backtest Simplifie', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Drawdown
ax2 = axes[1]
ax2.fill_between(range(len(drawdown)), drawdown * 100, alpha=0.5, color='coral')
ax2.set_xlabel('Periodes de rebalancement')
ax2.set_ylabel('Drawdown (%)')
ax2.set_title('Drawdown', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 50)
print("RESULTATS DU BACKTEST")
print("=" * 50)
print(f"  Rendement total:    {total_return:.2%}")
print(f"  Rendement annualise: {annualized_return:.2%}")
print(f"  Volatilite:         {vol:.2%}")
print(f"  Sharpe Ratio:       {sharpe:.3f}")
print(f"  Max Drawdown:       {max_dd:.2%}")

### Interpretation : Backtest simplifie

Le backtest simplifie compare la strategie Transformer (top quintile) a un benchmark equal-weight.

| Metrique | Signification |
|----------|---------------|
| **Rendement total** | Performance brute sur la periode de test |
| **Sharpe Ratio** | Rendement ajuste par le risque (>0.5 est bon) |
| **Max Drawdown** | Plus grosse perte depuis un pic |

**Points cles** :
1. Ce backtest est simplifie et ne tient pas compte des couts de transaction, slippage, ou contraintes de liquidite
2. Le top quintile represente une strategie long-only avec rotation hebdomadaire
3. Le Sharpe ratio est la metrique la plus importante : un Sharpe > 0.5 est exploitable en pratique
4. Pour un backtest realiste, utiliser QuantConnect avec les couts de transaction reels

---

## Section 12 : Sauvegarde du modele

Le modele est sauvegarde au format PyTorch avec les poids, le scaler et la configuration.

In [ ]:
# Sauvegarde du modele, scaler et configuration
import pickle
import io

model_path = 'transformer_multiasset_model.pt'

# Configuration du modele (pour reconstruction)
model_config = {
    'input_dim': n_features,
    'd_model': D_MODEL,
    'nhead': NHEAD,
    'num_layers': NUM_LAYERS,
    'dim_feedforward': DIM_FEEDFORWARD,
    'dropout': DROPOUT,
    'seq_len': SEQ_LEN,
    'pred_len': PRED_LEN,
    'feature_cols': feature_cols,
    'tickers': TICKERS_30,
}

# Sauvegarder le state_dict + scaler + config
save_dict = {
    'model_state_dict': model.state_dict(),
    'scaler': scaler,
    'config': model_config,
    'test_metrics': {
        'mse': mse,
        'mae': mae,
        'correlation': correlation,
        'dir_acc': dir_acc
    }
}

torch.save(save_dict, model_path)

# Verifier la taille
model_size = os.path.getsize(model_path)
print(f"Modele sauvegarde: {model_path}")
print(f"Taille: {model_size / 1024 / 1024:.2f} MB")
print(f"Compatible ObjectStore QC (<9 MB): {'Oui' if model_size < 9*1024*1024 else 'Non'}")

# Afficher le contenu
print(f"\nContenu sauvegarde:")
print(f"  model_state_dict: {len(save_dict['model_state_dict'])} couches")
print(f"  scaler: StandardScaler ({n_features} features)")
print(f"  config: {model_config}")
print(f"  test_metrics: {save_dict['test_metrics']}")

### Interpretation : Sauvegarde

Le modele est sauvegarde avec tout le necessaire pour le deploiement :

| Element | Contenu | Necessaire pour |
|---------|---------|----------------|
| `model_state_dict` | Poids du Transformer | Inference |
| `scaler` | StandardScaler fitte | Preprocessing des donnees |
| `config` | Hyperparametres + features | Reconstruction de l'architecture |
| `test_metrics` | Metriques de test | Documentation |

**Points cles** :
1. Le fichier doit peser moins de 9 MB pour etre compatible avec QuantConnect ObjectStore
2. La config permet de reconstruire exactement l'architecture sans hardcoder les parametres
3. Le scaler doit absolument etre conserve : les donnees de production doivent etre normalisees avec les memes parametres

---

## Section 13 : Integration QuantConnect Cloud

### Architecture de deploiement

```
LOCAL (GPU)
  |-- Entrainement du Transformer sur 10 ans de donnees
  |-- Sauvegarde state_dict + scaler + config
  \-- Upload vers ObjectStore QC

QUANTCONNECT CLOUD (CPU)
  |-- Chargement depuis ObjectStore
  |-- Reconstruction de l'architecture
  |-- Inference quotidienne (~100ms)
  \-- Generation d'insights (signaux de trading)
```

Le code suivant est une reference a copier dans `main.py` d'un projet QC Lab.

In [ ]:
# [REFERENCE QC] Code a copier dans main.py QC Lab (non executable ici)
# Code complet pour la strategie Transformer Multi-Asset sur QuantConnect

qc_transformer_strategy = '''from AlgorithmImports import *
import torch
import torch.nn as nn
import numpy as np
import pickle
import io
import math
from collections import deque


# ============================================
# TRANSFORMER MODEL DEFINITION
# ============================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class TransformerPredictor(nn.Module):
    def __init__(self, input_dim, d_model=256, nhead=8, num_layers=4,
                 dim_feedforward=1024, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
            activation='gelu'
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers,
            enable_nested_tensor=False
        )
        self.reg_head = nn.Sequential(
            nn.Linear(d_model, d_model // 4),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model // 4, 1)
        )
        self.cls_head = nn.Sequential(
            nn.Linear(d_model, d_model // 4),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model // 4, 1)
        )
    
    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)
        return self.reg_head(x), self.cls_head(x)


# ============================================
# ALPHA MODEL
# ============================================

class TransformerAlphaModel(AlphaModel):
    """Alpha Model utilisant un Transformer Encoder multi-actifs."""
    
    def __init__(self, algorithm, model_key="models/transformer_multiasset"):
        self.algorithm = algorithm
        self.model_key = model_key
        self.lookback = 60
        self.model = None
        self.scaler = None
        self.config = None
        self.symbol_data = {}
        self._load_model()
    
    def _load_model(self):
        if not self.algorithm.ObjectStore.ContainsKey(self.model_key):
            self.algorithm.Debug(f"Modele non trouve: {self.model_key}")
            return
        try:
            data = self.algorithm.ObjectStore.ReadBytes(self.model_key)
            checkpoint = torch.load(io.BytesIO(bytes(data)), map_location="cpu")
            self.config = checkpoint['config']
            self.scaler = checkpoint['scaler']
            self.model = TransformerPredictor(
                input_dim=self.config['input_dim'],
                d_model=self.config['d_model'],
                nhead=self.config['nhead'],
                num_layers=self.config['num_layers'],
                dim_feedforward=self.config['dim_feedforward'],
                dropout=self.config['dropout']
            )
            self.model.load_state_dict(checkpoint['model_state_dict'])
            self.model.eval()
            self.algorithm.Debug("Transformer model loaded from ObjectStore")
        except Exception as e:
            self.algorithm.Debug(f"Error loading model: {e}")
    
    def _prepare_features(self, symbol):
        history = self.algorithm.History(symbol, self.lookback + 25, Resolution.Daily)
        if history.empty or len(history) < self.lookback:
            return None
        try:
            close = history["close"].values
            features = []
            for i in range(self.lookback):
                idx = len(close) - self.lookback + i
                ret_1d = (close[idx] / close[idx-1] - 1) if idx > 0 else 0
                ret_5d = (close[idx] / close[max(0,idx-5)] - 1) if idx >= 5 else ret_1d
                ret_20d = (close[idx] / close[max(0,idx-20)] - 1) if idx >= 20 else ret_5d
                features.append([close[idx], ret_1d, ret_5d, ret_20d, 0, 0, 0, 50, 20, 0, 0, 0, 0])
            features = np.array(features, dtype=np.float32)
            features = self.scaler.transform(features)
            return features
        except Exception as e:
            self.algorithm.Debug(f"Feature error: {e}")
            return None
    
    def Update(self, algorithm, data):
        insights = []
        if self.model is None:
            return insights
        for symbol in algorithm.ActiveSecurities.Keys:
            if not data.ContainsKey(symbol):
                continue
            features = self._prepare_features(symbol)
            if features is None:
                continue
            with torch.no_grad():
                x = torch.FloatTensor(features).unsqueeze(0)
                reg_out, cls_out = self.model(x)
                direction_prob = torch.sigmoid(cls_out).item()
                predicted_return = reg_out.item()
            if direction_prob > 0.55:
                insights.append(Insight.Price(
                    symbol, timedelta(days=5), InsightDirection.Up,
                    magnitude=abs(predicted_return), confidence=direction_prob
                ))
            elif direction_prob < 0.45:
                insights.append(Insight.Price(
                    symbol, timedelta(days=5), InsightDirection.Down,
                    magnitude=abs(predicted_return), confidence=1-direction_prob
                ))
        return insights
    
    def OnSecuritiesChanged(self, algorithm, changes):
        pass


# ============================================
# MAIN ALGORITHM
# ============================================

class TransformerMultiAssetStrategy(QCAlgorithm):
    """
    Strategie multi-actifs utilisant un Transformer Encoder.
    
    - Modele: TransformerEncoder (4 layers, d_model=256, nhead=8)
    - Features: 10 par action + 4 macro
    - Signal: Direction prob + rendement predit
    """
    
    def Initialize(self):
        self.SetStartDate(2022, 1, 1)
        self.SetEndDate(2024, 12, 31)
        self.SetCash(100000)
        
        tickers = [
            "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA",
            "META", "JPM", "V", "UNH", "JNJ"
        ]
        for ticker in tickers:
            self.AddEquity(ticker, Resolution.Daily)
        
        self.SetAlpha(TransformerAlphaModel(self))
        self.SetPortfolioConstruction(EqualWeightingPortfolioConstructionModel())
        self.SetExecution(ImmediateExecutionModel())
        self.SetRiskManagement(MaximumDrawdownPercentPerSecurity(0.05))
        self.SetWarmUp(65)
        self.Debug("TransformerMultiAssetStrategy initialized")
'''

print("TransformerMultiAssetStrategy defini pour QuantConnect")
print("\n" + "=" * 60)
print("ARCHITECTURE TRANSFORMER POUR QC")
print("=" * 60)
print(f"""
Modele:
  - d_model: {D_MODEL}
  - nhead: {NHEAD}
  - num_layers: {NUM_LAYERS}
  - dim_feedforward: {DIM_FEEDFORWARD}

Features ({n_features}):
  - Per-stock: close, ret_1d, ret_5d, ret_20d, vol_10d, vol_20d, momentum, rsi
  - Macro: vix_level, vix_change, yield_spread, yield_curve_slope

Signal:
  - Direction probability (sigmoid of cls_head)
  - Predicted return (reg_head output)
  - Threshold: 0.55 pour long, 0.45 pour short

Workflow de deploiement:
  1. Entrainement local (GPU) -> sauvegarde state_dict
  2. Upload vers ObjectStore QC (<9 MB)
  3. Reconstruction automatique dans AlphaModel
  4. Inference quotidienne en CPU (~100ms)
""")

### Interpretation : Integration QuantConnect

Le code de reference pour QuantConnect comprend trois composants principaux :

| Composant | Role |
|-----------|------|
| `PositionalEncoding` | Encodage positionnel sinusooidal |
| `TransformerPredictor` | Architecture complete du modele |
| `TransformerAlphaModel` | Generation d'insights depuis le modele charge |
| `TransformerMultiAssetStrategy` | Strategie QC complete avec universe |

**Points cles** :
1. Le modele est charge depuis ObjectStore (state_dict + scaler + config)
2. L'architecture est reconstruite a l'identique grace a la config sauvegardee
3. Les features macro sont simplifiees dans la version QC (pas d'acces direct au VIX intra-algo)
4. Le seuil de confiance (0.55/0.45) filtre les signaux faibles pour reduire le turnover

---

## Section 14 : Resume et conclusions

In [ ]:
# Resume final

print("=" * 70)
print("RESUME - TRANSFORMER ENCODER MULTI-ASSET")
print("=" * 70)

print(f"""
ARCHITECTURE
  Modele:       TransformerEncoder (PyTorch natif)
  d_model:      {D_MODEL}
  Tetes:         {NHEAD}
  Couches:       {NUM_LAYERS}
  FFN dim:       {DIM_FEEDFORWARD}
  Parametres:   {sum(p.numel() for p in model.parameters()):,}

DONNEES
  Actions:      {N_STOCKS} top SP500
  Macro:        VIX, 10Y yield, 30Y yield
  Features:     {n_features} (10 par action + 4 macro)
  Sequence:     {SEQ_LEN} jours -> Prediction: {PRED_LEN} jours
  Periode:      {START} a {END}

RESULTATS TEST
  MSE:                {mse:.6f}
  MAE:                {mae:.6f}
  Correlation:        {correlation:.4f}
  Direction Accuracy: {dir_acc:.2%}

ENTRAINEMENT
  Epochs:       {len(history['train_loss'])} (early stopping: patience={PATIENCE})
  Optimiseur:   AdamW (lr={LR}, weight_decay=0.01)
  Scheduler:    CosineAnnealingLR
  Loss:         HuberLoss + 0.5*BCELoss
  Batch size:   {BATCH_SIZE} (thermal-safe)
""")

# Tableau comparatif Transformer vs LSTM
print("COMPARAISON TRANSFORMER vs LSTM")
print("-" * 50)
comparison = """
| Aspect           | Transformer         | LSTM                |
|------------------|---------------------|---------------------|
| Parallelisme     | Total (O(1) steps)  | Sequentiel (O(n))   |
| Attention        | Globale             | Dernier etat        |
| Interpretabilite | Poids d'attention   | Boite noire         |
| Parametres       | ~1-2M               | ~200-500K           |
| Temps/epoch      | Plus rapide (GPU)   | Plus lent           |
| Memoire GPU      | O(n^2) attention    | O(n) lineaire       |
| Longueur max     | ~500 positions      | Illimite (theorique)|
| Macro features   | Naturel (concat)    | Possible (concat)   |
| Overfitting      | Risque plus eleve   | Risque plus faible  |
"""
print(comparison)

print("PROCHAINES ETAPES")
print("-" * 50)
print("  1. Executer le backtest complet sur QuantConnect Cloud")
print("  2. Comparer avec les resultats LSTM du QC-Py-30")
print("  3. Explorer l'architecture PatchTST (QC-Py-22)")
print("  4. Tester Mamba pour les longues sequences (QC-Py-23)")
print("  5. Continuer avec QC-Py-32 : DQN Training")

---

## Conclusion et prochaines etapes

### Recapitulatif

Ce notebook a couvert l'implementation complete d'un Transformer Encoder pour le trading multi-actifs :

| Etape | Contenu |
|-------|--------|
| **Donnees** | 30 actions SP500 + 3 indicateurs macro (VIX, taux 10Y, 30Y) |
| **Features** | 10 par action + 4 macro, total 14 features |
| **Modele** | TransformerEncoder natif PyTorch (4 couches, 8 tetes, d_model=256) |
| **Entrainement** | AdamW + CosineAnnealingLR + gradient clipping + early stopping |
| **Evaluation** | Correlation, direction accuracy, MSE, MAE |
| **Interpretablite** | Heatmap des poids d'attention |
| **Deploiement** | Sauvegarde + code QC Cloud reference |

### Points cles a retenir

1. **Le Transformer capture les dependances a longue distance** via l'attention globale, contrairement au LSTM qui est limite par son etat cache
2. **Les features macro ameliorent la generalisation** en fournissant un contexte de marche partage entre toutes les actions
3. **L'interpretabilite de l'attention** est un avantage pratique : on peut verifier quelles periodes historiques influencent les predictions
4. **Le risque d'overfitting est reel** : le Transformer a beaucoup plus de parametres que le LSTM et necessite une regularisation soignee

### Limitations

| Limitation | Mitigation |
|------------|------------|
| Complexite O(n^2) | Limiter la sequence a 60 jours |
| Memoire GPU | Batch size reduit, empty_cache entre epochs |
| Overfitting | Dropout, early stopping, weight decay |
| Cout de transaction non inclus | Backtester sur QuantConnect |

### Prochaines etapes

- **QC-Py-32** : DQN Training - Reinforcement Learning pour le trading
- **QC-Py-22** : PatchTST et architectures SOTA pour series temporelles
- **QC-Py-23** : State Space Models (Mamba) pour les longues sequences en O(n)

### Ressources

- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) - Vaswani et al., 2017
- [Time-Series-Library](https://github.com/thuml/Time-Series-Library) - Framework unifie Tsinghua
- [PyTorch Transformer docs](https://pytorch.org/docs/stable/nn.html#transformer-layers)

---

[< Retour au sommaire](../README.md) | [Suivant : DQN Training >](QC-Py-32-DQN-Training.ipynb)